# Petrophysics → Reservoir Model → NeqSim

End-to-end Colab example showing how well logs and seismic information become a reservoir model and then production-process inputs.

**LAS → petrophysics → SEG-Y / well tie → 3D properties → upscaling → OPM-style properties → reservoir rates → NeqSim**

The example is synthetic and self-contained so it can run without proprietary data. Replace the synthetic inputs with Volve or other open field data for a real workflow.

In [ ]:
!pip -q install lasio segyio gstools pyvista neqsim
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import lasio, segyio, gstools as gs
np.random.seed(42)
WORK=Path('/content/subsurface_to_process'); WORK.mkdir(exist_ok=True)

## 1. Well logs → petrophysical properties

Synthetic GR, RHOB, NPHI, RT and DT are written to LAS and read back with `lasio`. We derive Vsh, porosity, water saturation, permeability and a simple facies class.

In [ ]:
z=np.arange(1500.,2500.,0.5)
res=(z>1750)&(z<2350)
phi_true=np.clip(np.where(res,0.23,0.08)+np.random.normal(0,0.015,len(z)),0.03,0.34)
gr=np.where(res,38.,90.)+np.random.normal(0,5,len(z))
rhob=2.65-phi_true*(2.65-1.03)+np.random.normal(0,0.02,len(z))
nphi=phi_true+np.random.normal(0,0.015,len(z))
sw_true=np.clip(np.where(res,0.25,0.85)+np.random.normal(0,0.04,len(z)),0.08,1)
rt=0.08/(np.maximum(sw_true,.05)**2*np.maximum(phi_true,.03)**2)
dt=55+180*phi_true+np.random.normal(0,2.5,len(z))
las=lasio.LASFile()
for name,data,unit in [('DEPT',z,'m'),('GR',gr,'API'),('RHOB',rhob,'g/cc'),('NPHI',nphi,'v/v'),('RT',rt,'ohm.m'),('DT',dt,'us/ft')]: las.append_curve(name,data,unit=unit)
las_path=WORK/'synthetic_well.las'; las.write(las_path)
df=lasio.read(las_path).df().reset_index()
df['VSH']=np.clip((df.GR-25)/(100-25),0,1)
phid=(2.65-df.RHOB)/(2.65-1.03)
df['PHI']=np.clip(0.5*phid+0.5*df.NPHI,0.02,0.35)
df['SW']=np.clip(np.sqrt(0.08/(np.maximum(df.RT,1e-3)*np.maximum(df.PHI,.02)**2)),.05,1)
df['PERM_MD']=np.clip(2500*(df.PHI**3/np.maximum((1-df.PHI)**2,1e-6))*np.exp(-2.5*df.VSH),.05,3000)
df['FACIES']=np.where((df.VSH<.35)&(df.PHI>.16),'Sand',np.where(df.VSH<.6,'Silty sand','Shale'))
df[['DEPT','VSH','PHI','SW','PERM_MD','FACIES']].head()

In [ ]:
fig,ax=plt.subplots(1,4,figsize=(11,8),sharey=True)
for a,x,label in zip(ax,[df.GR,df.PHI,df.SW,df.PERM_MD],['GR','PHI','SW','PERM [mD]']):
    a.plot(x,df.DEPT); a.set_xlabel(label); a.invert_yaxis(); a.grid(alpha=.25)
ax[0].set_ylabel('Depth [m]'); plt.tight_layout(); plt.show()

## 2. SEG-Y + simple well tie

`DT + RHOB → Vp → acoustic impedance → reflection coefficient → synthetic seismic`. In a field workflow, checkshot/VSP data would provide the depth-time relationship.

In [ ]:
def ricker(t,f):
    a=(np.pi*f*t)**2; return (1-2*a)*np.exp(-a)
time=np.arange(0,1600,2.0); ntr=80
seis=np.zeros((ntr,len(time)),dtype=np.float32)
for i in range(ntr):
    seis[i]+=ricker((time-(650+.5*(i-40)))/1000,25)
    seis[i]-=.8*ricker((time-(1000-.3*(i-40)))/1000,22)
segy_path=WORK/'synthetic_line.sgy'; segyio.tools.from_array2D(str(segy_path),seis,format=5,dt=2000)
vp=304800/df.DT.values; ai=vp*(df.RHOB.values*1000)
rc=np.zeros_like(ai); rc[1:]=(ai[1:]-ai[:-1])/(ai[1:]+ai[:-1])
syn=np.convolve(rc,ricker(np.linspace(-.08,.08,101),25),mode='same')
fig,ax=plt.subplots(1,2,figsize=(10,5))
ax[0].imshow(seis.T,aspect='auto',cmap='gray',extent=[0,ntr-1,time[-1],time[0]]); ax[0].set_title('SEG-Y section'); ax[0].set_ylabel('TWT [ms]')
ax[1].plot(syn,df.DEPT); ax[1].invert_yaxis(); ax[1].set_title('Log-derived synthetic'); ax[1].set_ylabel('Depth [m]'); plt.tight_layout(); plt.show()

## 3. Structural model + 3D reservoir properties

The simulator needs properties in every cell, not only along wells. Here a top/base structure and correlated geostatistical field distribute `PORO`, `PERMX`, `PERMY`, `PERMZ`, `SW`, `NTG` and `SATNUM` in 3D.

In [ ]:
nx,ny,nz=30,30,12
xc=np.linspace(0,1000,nx); yc=np.linspace(0,1000,ny)
X,Y=np.meshgrid(xc,yc,indexing='ij')
top=1780+.035*X-.02*Y+20*np.sin(X/250); base=top+520+25*np.sin(Y/220)
xx,yy,kk=np.meshgrid(xc,yc,np.arange(nz),indexing='ij')
field=gs.SRF(gs.Gaussian(dim=3,var=1,len_scale=[300,220,4]),seed=2026)((xx,yy,kk))
phi0=float(df.loc[df.FACIES=='Sand','PHI'].mean()); sw0=float(df.loc[df.FACIES=='Sand','SW'].mean()); k0=max(float(df.loc[df.FACIES=='Sand','PERM_MD'].median()),1)
PORO=np.clip(phi0+.025*field,0.06,.32); SW=np.clip(sw0-.06*field,.08,.8)
PERMX=np.clip(np.exp(np.log(k0)+1.8*field+10*(PORO-phi0)),.1,5000); PERMY=.75*PERMX; PERMZ=.08*PERMX
NTG=np.clip(1-1.6*(SW-.2),.15,1); SATNUM=np.where(PORO>.18,1,2)
fig,ax=plt.subplots(1,3,figsize=(13,4)); k=nz//2
for a,v,t in zip(ax,[PORO[:,:,k],np.log10(PERMX[:,:,k]),SW[:,:,k]],['PORO','log10 PERMX','SW']):
    im=a.imshow(v.T,origin='lower',extent=[0,1000,0,1000]); plt.colorbar(im,ax=a); a.set_title(t)
plt.tight_layout(); plt.show()

## 4. Upscaling + OPM Flow style export

Fine-scale log/geology properties are upscaled to simulation cells. The resulting arrays map directly to Eclipse/OPM Flow keywords.

In [ ]:
factor=2; nzc=nz//factor
PORO_c=PORO.reshape(nx,ny,nzc,factor).mean(3); SW_c=SW.reshape(nx,ny,nzc,factor).mean(3); PERMX_c=PERMX.reshape(nx,ny,nzc,factor).mean(3); NTG_c=NTG.reshape(nx,ny,nzc,factor).mean(3)
tmp=PERMZ.reshape(nx,ny,nzc,factor); PERMZ_c=factor/np.sum(1/np.maximum(tmp,1e-12),axis=3)
def write_kw(name,arr,fmt='{:.6g}'):
    with open(WORK/f'{name}.inc','w') as f:
        f.write(name+'\n'); vals=np.asarray(arr).flatten(order='F')
        for i in range(0,len(vals),8): f.write(' '.join(fmt.format(v) for v in vals[i:i+8])+'\n')
        f.write('/\n')
for n,a in [('PORO',PORO_c),('PERMX',PERMX_c),('PERMY',.75*PERMX_c),('PERMZ',PERMZ_c),('SWATINIT',SW_c)]: write_kw(n,a)
write_kw('SATNUM',np.where(PORO_c>.18,1,2),'{:.0f}')
print('OPM/Eclipse property files:',sorted(p.name for p in WORK.glob('*.inc')))

## 5. Reservoir model → rates

A simple tank/PI model is used only to make the notebook self-contained. Replace this section with OPM Flow for a real dynamic simulation.

In [ ]:
dx=1000/nx; dy=1000/ny; thick=base-top; dz=np.repeat((thick/nzc)[:,:,None],nzc,axis=2)
PV=np.sum(dx*dy*dz*PORO_c*NTG_c)
days=np.arange(0,3651,30); p=np.zeros(len(days)); qo=np.zeros(len(days)); qw=np.zeros(len(days)); qg=np.zeros(len(days)); p[0]=280
PI=float(np.clip(25*(np.median(PERMX_c)/100)*(np.mean(thick)*.7/100)/1.2,100,6000))
for i in range(len(days)):
    qo[i]=PI*max(p[i]-120,0); wc=np.clip(.05+.55*(1-p[i]/280),.05,.65); qw[i]=qo[i]*wc/max(1-wc,1e-6); qg[i]=120*qo[i]
    if i<len(days)-1: p[i+1]=max(120,p[i]-qo[i]*1.25*(days[i+1]-days[i])/PV/1.2e-4)
prod=pd.DataFrame({'day':days,'pressure_bara':p,'oil_Sm3_d':qo,'water_Sm3_d':qw,'gas_Sm3_d':qg})
prod['wellhead_pressure_bara']=np.maximum(35,.55*prod.pressure_bara); prod['temperature_C']=45
prod.to_csv(WORK/'reservoir_to_process.csv',index=False)
prod.head()

## 6. Reservoir output → NeqSim surface process

Reservoir rates and pressure become process-model boundary conditions. The example creates a fluid, separator and export compressor.

In [ ]:
from neqsim.thermo import fluid
from neqsim.process import stream, separator, compressor, runProcess, clearProcess
r=prod.iloc[0]; f=fluid('srk')
for c,x in [('nitrogen',.01),('methane',.74),('ethane',.09),('propane',.06),('n-butane',.03),('n-pentane',.02),('n-hexane',.02),('n-heptane',.03)]: f.addComponent(c,x)
f.setMixingRule('classic'); f.setTemperature(float(r.temperature_C),'C'); f.setPressure(float(r.wellhead_pressure_bara),'bara'); f.setTotalFlowRate(max(float(r.gas_Sm3_d)/1e6,1e-4),'MSm3/day')
clearProcess(); inlet=stream('wellstream',f); sep=separator('HP separator',inlet); comp=compressor('export compressor',sep.getGasOutStream(),pres=120.0); runProcess()
print('Gas rate [MSm3/d]:',r.gas_Sm3_d/1e6); print('Compressor power [MW]:',comp.getPower()/1e6)

## 7. Interfaces between disciplines

| From | To | Typical data |
|---|---|---|
| Petrophysics | Geomodel | tops, facies, PHI, Sw, K |
| Seismic | Geomodel | horizons, faults, attributes |
| Geomodel | Reservoir model | grid, zones, facies |
| Geostatistics | Simulator | PORO, PERM, NTG, SATNUM |
| Reservoir simulator | Wells/process | qo, qg, qw, BHP/Pwh |
| NeqSim | Facility model | phase split, properties, power, constraints |

Next step: replace the synthetic data with **Volve LAS + SEG-Y**, and replace the tank model with **OPM Flow**.